# SSS Marine Debris Detection — Stage 2 Training

## Stage 1 Results (Poor)
| Model | mAP50 | P | R | Problem |
|-------|-------|---|---|--------|
| YOLOv8n | 0.095 | 0.96 | 0.1 | Class imbalance → predicts nothing |
| YOLOv8-ESI | 0.06 | 0.52 | 0.1 | Same class imbalance |
| SS-YOLO | 0.0 | — | — | GhostConv/FastC2f can't transfer from YOLOv8n |

## Stage 2 Fixes
1. **SS-YOLO trains from scratch** — GhostConv/FastC2f have incompatible weight shapes with YOLOv8n
2. **Class imbalance fixed** — Debris oversampled 3x via augmentation (153→~460 debris images)
3. **Lower conf threshold** — 0.1 instead of 0.25 for evaluation (better recall)
4. **More epochs** — 150 epochs, higher LR for SS-YOLO from-scratch
5. **Copy-paste augmentation** — Paste debris onto background images

## Dataset: E5 Balanced
- **Train**: ~1260 images (460 debris + 800 clean BG + 600 noisy BG)
- **Val/Test**: Clean (no augmentation) for fair evaluation

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: Setup
# ═══════════════════════════════════════════════════════════
!pip install ultralytics pandas matplotlib -q
!git clone https://github.com/Dinoman67/sonarvision.git
%cd sonarvision

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 2: Upload E5 dataset and balance it
# ═══════════════════════════════════════════════════════════
from google.colab import files
uploaded = files.upload()  # Upload e5.zip
!unzip -q e5.zip -d /content/

# Balance the dataset — oversample debris 3x
!python scripts/balance_e5_dataset.py \
    --e5 /content/e5 \
    --output /content/e5_balanced \
    --ratio 3

# Verify balanced dataset
import os
print('\nBalanced dataset stats:')
for split in ['train', 'val', 'test']:
    imgs = [f for f in os.listdir(f'/content/e5_balanced/images/{split}') if f.endswith('.png')]
    lbls = [f for f in os.listdir(f'/content/e5_balanced/labels/{split}') if f.endswith('.txt')]
    pos = sum(1 for l in lbls if os.path.getsize(f'/content/e5_balanced/labels/{split}/{l}') > 0)
    aug = sum(1 for i in imgs if '_A' in i)
    noisy = sum(1 for i in imgs if '_N' in i)
    print(f'  {split}: {len(imgs)} images ({pos} debris, {len(imgs)-pos} bg, {aug} augmented, {noisy} noisy)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 3: Visualize balanced dataset
# ═══════════════════════════════════════════════════════════
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

train_dir = '/content/e5_balanced/images/train'

# Show original vs augmented debris
debris_orig = sorted([f for f in os.listdir(train_dir) if 'TGT' in f and '_A' not in f and '_N' not in f])[:3]
debris_aug = sorted([f for f in os.listdir(train_dir) if 'TGT' in f and '_A' in f])[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, name in enumerate(debris_orig):
    axes[0, i].imshow(np.array(Image.open(os.path.join(train_dir, name))), cmap='gray')
    axes[0, i].set_title(f'Original: {name[:25]}', fontsize=9)
    axes[0, i].axis('off')
for i, name in enumerate(debris_aug[:3]):
    axes[1, i].imshow(np.array(Image.open(os.path.join(train_dir, name))), cmap='gray')
    axes[1, i].set_title(f'Augmented: {name[:25]}', fontsize=9)
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=11, rotation=0, labelpad=80)
axes[1, 0].set_ylabel('Augmented', fontsize=11, rotation=0, labelpad=80)
plt.suptitle('Debris Oversampling: Original vs Augmented', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## STEP 4: Train All Three Models

Key changes from Stage 1:
- **SS-YOLO**: Trains from scratch (no pretrained weights)
- **All models**: 150 epochs, lower LR for fine-tuning
- **Evaluation**: conf=0.1 (lower threshold for better recall)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Load custom modules
# ═══════════════════════════════════════════════════════════
import torch.nn as nn
from ultralytics import YOLO
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules.block import C2f, SPPF
from models.sss_custom_modules import (
    WaveletConv, FastC2f, GhostConv,
    PConv, FasterBlock, DepthwiseSeparableConv, SEBlock, CBAM\n)
from models.build_sss_models import build_ss_yolo, build_yolov8_esi_full, C2fWithSE
from ultralytics.models.yolo.detect.train import DetectionTrainer
print('✓ Custom modules loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Helper: Confidence threshold sweep
# ═══════════════════════════════════════════════════════════
def conf_sweep(model, data_yaml, imgsz=512, confs=None):
    """Evaluate model across confidence thresholds."""
    if confs is None:
        confs = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
    print(f"  {'Conf':>6} {'P':>8} {'R':>8} {'mAP50':>8} {'F1':>8}")
    print(f"  {'-'*42}")
    best_f1 = 0
    best_conf = 0.25
    for c in confs:
        r = model.val(data=data_yaml, imgsz=imgsz, conf=c, verbose=False)
        p, rv = r.box.mp, r.box.mr
        f1 = 2*p*rv / max(p+rv, 1e-8)
        marker = ' ←' if f1 > best_f1 else ''
        print(f"  {c:>6.2f} {p:>8.4f} {rv:>8.4f} {r.box.map50:>8.4f} {f1:>8.4f}{marker}")
        if f1 > best_f1:
            best_f1 = f1
            best_conf = c
    return best_conf, best_f1

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train YOLOv8n on E5 Balanced
# ═══════════════════════════════════════════════════════════
DATA = '/content/e5_balanced/data.yaml'
IMGSZ = 512

print('\n' + '='*60)
print('Training: YOLOv8n (Baseline) on E5 Balanced')
print('='*60)

model_v8n = YOLO('yolov8n.pt')
model_v8n.train(
    data=DATA, epochs=150, imgsz=IMGSZ, batch=16, patience=40,
    lr0=0.005, lrf=0.01, warmup_epochs=3,
    mosaic=0.0, mixup=0.0,
    fliplr=0.0, flipud=0.0, degrees=0.0,
    translate=0.05, scale=0.2,
    name='yolov8n_e5s2', project='/content/runs', exist_ok=True, plots=True,
)

# Evaluate with conf sweep
print('\n--- Confidence Sweep (val) ---')
best_conf_v8n, _ = conf_sweep(model_v8n, DATA)

val_v8n = model_v8n.val(data=DATA, split='val', conf=best_conf_v8n)
test_v8n = model_v8n.val(data=DATA, split='test', conf=best_conf_v8n)

print(f'\n✓ YOLOv8n on E5 Balanced (conf={best_conf_v8n}):')
print(f'  Val  — mAP50: {val_v8n.box.map50:.4f}, P: {val_v8n.box.mp:.4f}, R: {val_v8n.box.mr:.4f}')
print(f'  Test — mAP50: {test_v8n.box.map50:.4f}, P: {test_v8n.box.mp:.4f}, R: {test_v8n.box.mr:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train SS-YOLO on E5 Balanced (FROM SCRATCH)
# ═══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Training: SS-YOLO (GhostConv + FastC2f) FROM SCRATCH')
print('='*60)
print('NOTE: SS-YOLO trains from random init — GhostConv/FastC2f')
print('      have incompatible weight shapes with YOLOv8n.')

# Build SS-YOLO from scratch (pretrained=None)
ss_model = build_ss_yolo(pretrained=None)

_orig = DetectionTrainer.get_model
def _patched_ss(self, cfg=None, weights=None, verbose=True):
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils import RANK
    model = self.set_model_names_for_load(
        DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'], verbose=verbose and RANK == -1)
    )
    model.model = ss_model.model
    model.nc = 1
    model.names = {0: 'marine_debris'}
    # Do NOT load pretrained weights — SS-YOLO is from scratch
    return model
DetectionTrainer.get_model = _patched_ss

try:
    yolo_ss = YOLO('yolov8n.pt')  # Template for YAML only
    yolo_ss.train(
        data=DATA,
        epochs=200,  # More epochs for from-scratch training
        imgsz=IMGSZ, batch=16, patience=50,
        lr0=0.01, lrf=0.05, warmup_epochs=5,  # Higher LR for scratch
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='ss_yolo_e5s2', project='/content/runs', exist_ok=True, plots=True,
    )

    print('\n--- Confidence Sweep (val) ---')
    best_conf_ss, _ = conf_sweep(yolo_ss, DATA)

    val_ss = yolo_ss.val(data=DATA, split='val', conf=best_conf_ss)
    test_ss = yolo_ss.val(data=DATA, split='test', conf=best_conf_ss)
    print(f'\n✓ SS-YOLO on E5 Balanced (conf={best_conf_ss}):')
    print(f'  Val  — mAP50: {val_ss.box.map50:.4f}, P: {val_ss.box.mp:.4f}, R: {val_ss.box.mr:.4f}')
    print(f'  Test — mAP50: {test_ss.box.map50:.4f}, P: {test_ss.box.mp:.4f}, R: {test_ss.box.mr:.4f}')
except Exception as e:
    print(f'✗ SS-YOLO failed: {e}')
    import traceback; traceback.print_exc()
finally:
    DetectionTrainer.get_model = _orig

In [ ]:
# ═══════════════════════════════════════════════════════════
# Train YOLOv8-ESI on E5 Balanced
# ═══════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Training: YOLOv8-ESI (SE Attention) on E5 Balanced')
print('='*60)

esi_model = build_yolov8_esi_full()

_orig2 = DetectionTrainer.get_model
def _patched_esi(self, cfg=None, weights=None, verbose=True):
    from ultralytics.nn.tasks import DetectionModel
    from ultralytics.utils import RANK
    model = self.set_model_names_for_load(
        DetectionModel(cfg, nc=self.data['nc'], ch=self.data['channels'], verbose=verbose and RANK == -1)
    )
    model.model = esi_model.model
    model.nc = 1
    model.names = {0: 'marine_debris'}
    # Load pretrained weights — ESI keeps Conv/C2f, so transfer works
    try:
        model.load(weights)
    except Exception as e:
        print(f'  ⚠ Weight loading: {e}')
    return model
DetectionTrainer.get_model = _patched_esi

try:
    yolo_esi = YOLO('yolov8n.pt')
    yolo_esi.train(
        data=DATA, epochs=150, imgsz=IMGSZ, batch=16, patience=40,
        lr0=0.005, lrf=0.01, warmup_epochs=3,
        mosaic=0.0, mixup=0.0,
        fliplr=0.0, flipud=0.0, degrees=0.0,
        translate=0.05, scale=0.2,
        name='yolov8_esi_e5s2', project='/content/runs', exist_ok=True, plots=True,
    )

    print('\n--- Confidence Sweep (val) ---')
    best_conf_esi, _ = conf_sweep(yolo_esi, DATA)

    val_esi = yolo_esi.val(data=DATA, split='val', conf=best_conf_esi)
    test_esi = yolo_esi.val(data=DATA, split='test', conf=best_conf_esi)
    print(f'\n✓ YOLOv8-ESI on E5 Balanced (conf={best_conf_esi}):')
    print(f'  Val  — mAP50: {val_esi.box.map50:.4f}, P: {val_esi.box.mp:.4f}, R: {val_esi.box.mr:.4f}')
    print(f'  Test — mAP50: {test_esi.box.map50:.4f}, P: {test_esi.box.mp:.4f}, R: {test_esi.box.mr:.4f}')
except Exception as e:
    print(f'✗ YOLOv8-ESI failed: {e}')
    import traceback; traceback.print_exc()
finally:
    DetectionTrainer.get_model = _orig2

## STEP 5: Compare Results

In [ ]:
import pandas as pd

results = []
model_map = {'YOLOv8n': model_v8n, 'SS-YOLO': yolo_ss, 'YOLOv8-ESI': yolo_esi}
val_map = {'YOLOv8n': val_v8n, 'SS-YOLO': val_ss, 'YOLOv8-ESI': val_esi}
test_map = {'YOLOv8n': test_v8n, 'SS-YOLO': test_ss, 'YOLOv8-ESI': test_esi}
conf_map = {'YOLOv8n': best_conf_v8n, 'SS-YOLO': best_conf_ss, 'YOLOv8-ESI': best_conf_esi}

for name in ['YOLOv8n', 'SS-YOLO', 'YOLOv8-ESI']:
    val_r, test_r = val_map[name], test_map[name]
    p, r = test_r.box.mp, test_r.box.mr
    f1 = 2*p*r / max(p+r, 1e-8)
    n = sum(p_.numel() for p_ in model_map[name].model.parameters())
    results.append({
        'Model': name, 'Params': f'{n/1e6:.2f}M',
        'Best Conf': f'{conf_map[name]:.2f}',
        'Val mAP50': f'{val_r.box.map50:.4f}',
        'Test mAP50': f'{test_r.box.map50:.4f}',
        'Test P': f'{p:.4f}', 'Test R': f'{r:.4f}',
        'Test F1': f'{f1:.4f}',
    })

df = pd.DataFrame(results)
print('\n' + '='*70)
print('STAGE 2 RESULTS — E5 Balanced Dataset')
print('='*70)
print(df.to_string(index=False))

# Comparison with Stage 1
print('\n' + '='*70)
print('STAGE 1 vs STAGE 2 COMPARISON')
print('='*70)
print('Model          Stage 1 mAP50  Stage 2 mAP50  Improvement')
print('-'*55)
stage1 = {'YOLOv8n': 0.095, 'SS-YOLO': 0.0, 'YOLOv8-ESI': 0.06}
for name in ['YOLOv8n', 'SS-YOLO', 'YOLOv8-ESI']:
    s1 = stage1[name]
    s2 = float(df.loc[df['Model'] == name, 'Test mAP50'].values[0])
    delta = s2 - s1
    arrow = '↑' if delta > 0 else '↓' if delta < 0 else '='
    print(f'{name:<15} {s1:>12.3f} {s2:>12.3f}  {arrow} {delta:+.3f}')

df.to_csv('/content/results_e5_stage2.csv', index=False)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Per-target breakdown
# ═══════════════════════════════════════════════════════════
print('\n' + '='*70)
print('PER-TARGET DETECTION (test set)')
print('='*70)

for name, model in [('YOLOv8n', model_v8n), ('SS-YOLO', yolo_ss), ('YOLOv8-ESI', yolo_esi)]:
    conf = conf_map[name]
    print(f'\n--- {name} (conf={conf}) ---')
    preds = model.predict(source='/content/e5_balanced/images/test', imgsz=IMGSZ,
                          conf=conf, save=False, verbose=False)
    
    target_stats = {}
    for r in preds:
        img_name = os.path.basename(str(r.path))
        parts = img_name.replace('.png','').split('_')
        tid = 'BG'
        for p in parts:
            if p.startswith('TGT'):
                tid = p
                break
        
        if tid not in target_stats:
            target_stats[tid] = {'detected': 0, 'total': 0, 'confs': []}
        target_stats[tid]['total'] += 1
        if len(r.boxes) > 0:
            target_stats[tid]['detected'] += 1
            target_stats[tid]['confs'].extend([float(c) for c in r.boxes.conf])
    
    print(f"  {'Target':>8} {'Imgs':>5} {'Det':>4} {'Rate':>7} {'AvgConf':>8}")
    for tid in sorted(target_stats.keys()):
        s = target_stats[tid]
        rate = s['detected'] / max(s['total'], 1)
        avg_c = np.mean(s['confs']) if s['confs'] else 0
        bar = '█' * int(rate * 15) + '░' * (15 - int(rate * 15))
        print(f'  {tid:>8} {s["total"]:>5} {s["detected"]:>4} {bar} {rate:>6.0%} {avg_c:>7.3f}')

## STEP 6: Export Best Model

In [ ]:
# ═══════════════════════════════════════════════════════════
# Export best model
# ═══════════════════════════════════════════════════════════
result_df = pd.read_csv('/content/results_e5_stage2.csv')
# Find best by F1
f1_vals = []
for name in result_df['Model']:
    test_r = test_map[name]
    p, r = test_r.box.mp, test_r.box.mr
    f1_vals.append(2*p*r / max(p+r, 1e-8))
result_df['F1'] = f1_vals

best_row = result_df.loc[result_df['F1'].idxmax()]
best_name = best_row['Model']
best_model = model_map[best_name]

print(f'Best model: {best_name} (F1={best_row["F1"]:.4f})')
best_model.export(format='onnx', imgsz=IMGSZ)

from google.colab import files
# Map model name to directory
dir_map = {'YOLOv8n': 'yolov8n_e5s2', 'SS-YOLO': 'ss_yolo_e5s2', 'YOLOv8-ESI': 'yolov8_esi_e5s2'}
model_dir = f'/content/runs/{dir_map[best_name]}'
files.download(f'{model_dir}/weights/best.pt')
files.download(f'{model_dir}/weights/best.onnx')